In [1]:
from platform import python_version
print(python_version())

3.12.12


## A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma (study, forked)

Huoshen W, Yuan K, Xiong J, Lin Y, Yu W, Xie Y, Yuan Q, Zhang X, Dong C, Sun C, Yi S. A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma through bulk to single-cell and spatial transcriptomics integration. Hum Genomics. 2026 Mar 16;20(1):77. doi: 10.1186/s40246-026-00919-w. PMID: 41840725; PMCID: PMC13104449.

https://pubmed.ncbi.nlm.nih.gov/41840725/

### Claude analysis - 2 studies

### does these studies have accessible data to calculate spatial transcriptomics and scVelo?

1. doi: 10.1002/smmd.70023. eCollection 2025 Dec. - Integrating Single-Cell and Spatial Transcriptomics Reveals NK Cell Subpopulations Associated With Immunotherapy for Melanoma
2. doi: 10.1186/s40246-026-00919-w. - A cross-scale multimodal framework identifies clinically actionable immunotherapy biomarkers in melanoma through bulk to single-cell and spatial transcriptomics integration



Both papers are pure re-analyses — neither deposited any new data. 
1. Paper 1's availability statement points to the corresponding author on request; 
2. Paper 2 points to the datasets listed in its Additional file 1, Table S1. So the question becomes whether the accessions they reuse are accessible, which I checked one by one in GEO/SRA/ENA.

**Short answer: spatial yes for both; scVelo only on the TIL/lymphocyte scRNA datasets, not on the melanoma atlases either paper is built around.**

#### Paper 1 — NK subpopulations (10.1002/smmd.70023)

**Spatial (GSE229532) — accessible, but it is mouse.** The record is a Visium V1 fresh-frozen experiment on BRAF^V600E/PTEN-deficient **Mus musculus** melanomas (2 samples, subseries of GSE200751), with processed outputs public: `filtered_feature_bc_matrix.h5`, tissue positions, hi-res images, plus annotated and clustered Seurat objects, and 2 raw runs in ENA. Worth flagging: the paper describes this ST data as accessed through UCSC Xena and sourced from TCGA, which does not match what GSE229532 is — TCGA has no spatial data, and this series is a mouse Visium experiment. Whatever the intended provenance, what you can download and re-run is mouse tumour tissue, not human skin melanoma.

**scVelo — only one of their three scRNA inputs qualifies.**
- GSE72056 (Tirosh): all 3,247 SRA runs return protected-file/dbGaP notices — controlled access, no velocity.
- GSE115978 (Jerby-Arnon): counts, TPM and annotations only; no raw reads deposited anywhere.
- GSE242477 (acral melanoma TILs): 17 runs public, 10x 5′ with the 26–28 bp barcode read present → **scVelo feasible**, but this is sorted T cells, so velocity runs on lymphocytes, not malignant cells.

#### Paper 2 — cross-scale multimodal (10.1186/s40246-026-00919-w)

**Spatial — both sources accessible, and the melanoma one is human.** GSM5420750, their brain-metastasis sample, is the single melanoma-primary section in GSE179572 (the other five are lung adenocarcinoma, renal cell and breast); Visium V1 on OCT fresh-frozen tissue, spaceranger 1.2.0, with matrices, spot positions and images public and 12 ENA runs carrying FASTQ (R1 = 28 bp barcode+UMI, R2 = 90 bp, plus dual index). Their second ST source, the 10x "Human Melanoma, IF Stained (FFPE)" demo, is freely downloadable from 10x.

**scVelo — one of three scRNA inputs.**
- GSE148190 (melanoma TIL/LN/blood, 10x 5′ GEX + VDJ): 24 GEX runs public at 2×150; I pulled the first reads and confirmed barcode+UMI followed by the 5′ TSO in R1 → **scVelo feasible**. Again lymphocytes only.
- GSE215120 (acral melanoma scRNA): CellRanger matrices only, no single-cell raw reads — the superseries' public raw is bulk SMARTer RNA-seq.
- GSE151091 (human melanocytes): 48 ENA runs exist, but both reads are ~100 bp cDNA with no barcode read and 6 runs have no FASTQ at all; GEO gives only a series-level count CSV. Not reconstructible — and it is normal melanocytes, not tumour.

## Practical upshot

Spatial re-analysis is straightforward in both cases: the processed Visium outputs load directly with `Load10X_Spatial`/`squidpy`, and for GSM5420750 and GSE229532 the raw reads are there too if you want to re-run spaceranger. Velocity is the constraint. Two caveats worth weighing before committing:

1. **The FFPE Visium dataset can never support velocity** — probe-based capture generates no intronic reads, so there is no unspliced fraction to estimate, independent of data access.
2. **Velocity on Visium is spot-level**, 55 µm spots containing several cells, so "spliced/unspliced" mixes cell types within a spot; both fresh-frozen Visium datasets (GSM5420750, GSE229532) are chemically compatible via velocyto on the spaceranger BAM, but the biological reading is weaker than a single-cell result.

If the aim is velocity in malignant melanoma cells with spatial context from the same study, neither paper provides it — their velocity-capable accessions are T-cell products. The unsorted whole-tumour options with usable raw reads remain the ones from the earlier shortlist (GSE269936, GSE222446, GSE189889, GSE174401), which you would have to pair with GSM5420750 or an independent melanoma Visium series for the spatial side.

[paper_data_availability_spatial_scvelo.csv]({{artifact:e3581a8a-8b79-420a-81e1-3729e6622391}}) — every accession cited by the two papers with assay, organism, exactly what is public, and the spatial/scVelo verdict per dataset.

In [2]:
import os, sys, re, yaml
from pathlib import Path
from dotenv import load_dotenv
import time
import requests
import json

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

from scipy.stats import spearmanr

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

ROOT0 = Path("/home/flavio/uv/rna_velocity")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import create_dir, pdreadcsv, pdwritecsv

root_figure = create_dir(ROOT0, "pictures")
root_results = create_dir(ROOT0, "results")
root_tables = create_dir(root_results, "tables")

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/rna_velocity
ROOT_SRC added: /home/flavio/uv/rna_velocity/src


In [3]:
root_search = create_dir(ROOT0, "download/search")
os.listdir(root_search)

['paper_data_availability_spatial_scvelo.csv']

In [4]:
fname = 'paper_data_availability_spatial_scvelo.csv'

df = pdreadcsv(fname, root_search, sep=',')
df

,doi,paper,accession,assay,organism,n_samples,what_is_public,spatial_reanalysis,scvelo_feasible,notes
0,10.1002/smmd.70023,NK subpopulations (Smart Med 2025),GSE229532,"spatial (Visium V1, fresh-frozen)",Mus musculus,2,processed h5 + tissue positions + hires images + Seurat RData; 2 ENA runs wi...,yes - spatial re-analysis fully possible,spot-level velocity possible (poly-A chemistry),"MOUSE BRAF-V600E/PTEN tumours, subseries of GSE200751; paper describes it as..."
1,10.1002/smmd.70023,NK subpopulations,GSE72056,scRNA (Smart-seq2),Homo sapiens,4645,processed TPM matrix only; all 3247 SRA runs return 'Protected file(s). Go t...,NaN,NO - raw reads controlled access (dbGaP),Tirosh 2016 melanoma atlas
2,10.1002/smmd.70023,NK subpopulations,GSE115978,scRNA (Smart-seq2),Homo sapiens,7186,counts + TPM + annotations only; no SRA runs deposited,NaN,NO - no raw reads in any archive,Jerby-Arnon 2018 ICI-resistance atlas
3,10.1002/smmd.70023,NK subpopulations,GSE242477,scRNA 10x 5' GEX + VDJ,Homo sapiens,17,CellRanger matrices + 17 ENA runs with FASTQ (26-28 bp barcode read),NaN,YES,"acral melanoma TILs - T cells only, no malignant cells"
4,10.1186/s40246-026-00919-w,Cross-scale multimodal (Hum Genomics 2026),GSE179572 / GSM5420750,"spatial (Visium V1, fresh-frozen OCT, spaceranger 1.2.0)",Homo sapiens,1,processed h5/mtx + positions + images; 12 ENA runs with FASTQ (R1 28 bp barc...,YES - the melanoma brain-metastasis section,spot-level velocity possible (poly-A chemistry),GSM5420750 is the only melanoma-primary sample; other 5 are lung/RCC/breast
5,10.1186/s40246-026-00919-w,Cross-scale multimodal,"10x 'Human Melanoma, IF Stained (FFPE)'","spatial (Visium FFPE, probe-based)",Homo sapiens,1,"public 10x Genomics demo dataset (matrices, images, FASTQ)",YES - spatial re-analysis possible,NO - probe-based FFPE capture has no intronic/unspliced signal,primary melanoma
6,10.1186/s40246-026-00919-w,Cross-scale multimodal,GSE215120,scRNA (acral melanoma),Homo sapiens,11,CellRanger matrices only; no SRA runs (superseries' public raw is bulk SMART...,NaN,NO - no single-cell raw reads public,subseries of GSE215121
7,10.1186/s40246-026-00919-w,Cross-scale multimodal,GSE148190,scRNA 10x 5' GEX + VDJ,Homo sapiens,14,"matrices + integrated .rds; 24 GEX runs with FASTQ, 2x150, barcode+UMI+TSO c...",NaN,YES,"melanoma TIL/LN/blood - lymphocytes, no malignant cells"
8,10.1186/s40246-026-00919-w,Cross-scale multimodal,GSE151091,scRNA (human melanocytes),Homo sapiens,22,"series-level raw count CSV only; 48 ENA runs but both reads ~100 bp cDNA, no...",NaN,NO - no cell-barcode read recoverable,"normal skin melanocytes, not tumour"


In [6]:
cols = ['doi', 'paper', 'accession', 'assay', 'organism', 'n_samples', 'what_is_public',
       'spatial_reanalysis', 'scvelo_feasible', 'notes']
cols = ['accession', 'paper',  'assay', 'n_samples', 'what_is_public',
       'spatial_reanalysis', 'scvelo_feasible', 'notes']

df[df.organism == 'Homo sapiens'][cols]

,accession,paper,assay,n_samples,what_is_public,spatial_reanalysis,scvelo_feasible,notes
1,GSE72056,NK subpopulations,scRNA (Smart-seq2),4645,processed TPM matrix only; all 3247 SRA runs return 'Protected file(s). Go t...,NaN,NO - raw reads controlled access (dbGaP),Tirosh 2016 melanoma atlas
2,GSE115978,NK subpopulations,scRNA (Smart-seq2),7186,counts + TPM + annotations only; no SRA runs deposited,NaN,NO - no raw reads in any archive,Jerby-Arnon 2018 ICI-resistance atlas
3,GSE242477,NK subpopulations,scRNA 10x 5' GEX + VDJ,17,CellRanger matrices + 17 ENA runs with FASTQ (26-28 bp barcode read),NaN,YES,"acral melanoma TILs - T cells only, no malignant cells"
4,GSE179572 / GSM5420750,Cross-scale multimodal (Hum Genomics 2026),"spatial (Visium V1, fresh-frozen OCT, spaceranger 1.2.0)",1,processed h5/mtx + positions + images; 12 ENA runs with FASTQ (R1 28 bp barc...,YES - the melanoma brain-metastasis section,spot-level velocity possible (poly-A chemistry),GSM5420750 is the only melanoma-primary sample; other 5 are lung/RCC/breast
5,"10x 'Human Melanoma, IF Stained (FFPE)'",Cross-scale multimodal,"spatial (Visium FFPE, probe-based)",1,"public 10x Genomics demo dataset (matrices, images, FASTQ)",YES - spatial re-analysis possible,NO - probe-based FFPE capture has no intronic/unspliced signal,primary melanoma
6,GSE215120,Cross-scale multimodal,scRNA (acral melanoma),11,CellRanger matrices only; no SRA runs (superseries' public raw is bulk SMART...,NaN,NO - no single-cell raw reads public,subseries of GSE215121
7,GSE148190,Cross-scale multimodal,scRNA 10x 5' GEX + VDJ,14,"matrices + integrated .rds; 24 GEX runs with FASTQ, 2x150, barcode+UMI+TSO c...",NaN,YES,"melanoma TIL/LN/blood - lymphocytes, no malignant cells"
8,GSE151091,Cross-scale multimodal,scRNA (human melanocytes),22,"series-level raw count CSV only; 48 ENA runs but both reads ~100 bp cDNA, no...",NaN,NO - no cell-barcode read recoverable,"normal skin melanocytes, not tumour"


### Start with GSE148190 (Apr 07, 2020)

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148190

**title**: Single cell RNA and TCR sequencing of tumor-infiltrating lymphocytes from human melanoma

**reference**: Mahuron KM, Moreau JM, Glasgow JE, Boda DP, Pauli ML, Gouirand V, Panjabi L, Grewal R, Luber JM, Mathur AN, Feldman RM, Shifrut E, Mehta P, Lowe MM, Alvarado MD, Marson A, Singer M, Wells J, Jupp R, Daud AI, Rosenblum MD. Layilin augments integrin activation to promote antitumor immunity. J Exp Med. 2020 Sep 7;217(9):e20192080. doi: 10.1084/jem.20192080. PMID: 32539073; PMCID: PMC7478725.

**features**: human, melanoma, n_samples=14, and scRNA 10x 5' GEX + VDJ

**platform**: GPL24676	Illumina NovaSeq 6000 (Homo sapiens)

https://seqout.org/p/SRP255479

In [7]:
def soft(acc, targ="self", view="brief"):
    try:
        text = requests.get("https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi",
        params={"acc": acc, "targ": targ, "form": "text", "view": view}, timeout=180).text
    except requests.RequestException as e:
        print(f"Error fetching data for {acc}: {e}")
        text = ""

    return text

gse = "GSE148190"

s = soft(gse)
g = soft(gse, targ="gsm", view="quick")

if s != '' and g != '':
    print("status/dates:", re.findall(r"!Series_(status|submission_date|last_update_date) = (.*)", s))
    print("type:", re.findall(r"!Series_type = (.*)", s), "| n GSM:", len(re.findall(r"!Series_sample_id", s)))
    print("platform:", re.findall(r"!Series_platform_id = (.*)", s), re.findall(r"!Series_pubmed_id = (.*)", s))
    rows = []
    for b in re.split(r"(?=\^SAMPLE = )", g):
        if not b.startswith("^SAMPLE"): continue
        rows.append({"gsm": re.search(r"\^SAMPLE = (GSM\d+)", b).group(1),
                    "title": (re.findall(r"!Sample_title = (.*)", b) or [""])[0].strip(),
                    "org": (re.findall(r"!Sample_organism_ch1 = (.*)", b) or [""])[0].strip(),
                    "chars": "; ".join(x.strip() for x in re.findall(r"!Sample_characteristics_ch1 = (.*)", b)),
                    "strategy": (re.findall(r"!Sample_library_strategy = (.*)", b) or [""])[0].strip(),
                    "proc": " ".join(re.findall(r"!Sample_data_processing = (.*)", b))[:120]})
    dfc = pd.DataFrame(rows)
    print("\nchemistry mentions:", sorted(set(re.findall(r"(?i)(5'|five.?prime|3'|V\(D\)J|VDJ|Chromium|Single Cell Immune Profiling|cellranger \S+|vdj)", g)))[:12])
else:
    dfc = pd.DataFrame()

# print(d[["gsm","title","chars","strategy"]].to_string(index=False))
dfc


status/dates: [('status', 'Public on Apr 07 2020\r'), ('submission_date', 'Apr 06 2020\r'), ('last_update_date', 'Feb 06 2025\r')]
type: ['Expression profiling by high throughput sequencing\r', 'Other\r'] | n GSM: 14
platform: ['GPL24676\r'] ['32539073\r']

chemistry mentions: ["5'", 'Chromium', 'V(D)J', 'VDJ']


,gsm,title,org,chars,strategy,proc
0,GSM4455931,melanoma_K383_LN_GEX,Homo sapiens,tissue: melanoma; tumor site: Involved regional lymph node; treatment status...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
1,GSM4455932,melanoma_K409_blood_GEX,Homo sapiens,tissue: PBMC; treatment status: naive; sort purification strategy: Cells wer...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
2,GSM4455933,melanoma_K409_LN_GEX,Homo sapiens,tissue: melanoma; tumor site: Involved regional lymph node; treatment status...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
3,GSM4455934,melanoma_K409_LN_VDJ,Homo sapiens,tissue: melanoma; tumor site: Involved regional lymph node; treatment status...,OTHER,"Library strategy: TCR-Seq\r Cell Ranger analysis pipelines (version 3.0.2, 1..."
4,GSM4455935,melanoma_K409_tumor_GEX,Homo sapiens,tissue: melanoma; tumor site: Cutaneous primary tumor; treatment status: nai...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
5,GSM4455936,melanoma_K409_tumor_VDJ,Homo sapiens,tissue: melanoma; tumor site: Cutaneous primary tumor; treatment status: nai...,OTHER,"Library strategy: TCR-Seq\r Cell Ranger analysis pipelines (version 3.0.2, 1..."
6,GSM4455937,melanoma_K411_blood_GEX,Homo sapiens,tissue: PBMC; treatment status: naive; sort purification strategy: Cells wer...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
7,GSM4455938,melanoma_K411_LN_GEX,Homo sapiens,tissue: melanoma; tumor site: Involved regional lymph node; treatment status...,RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
8,GSM8774026,K374_Tumor1_GEX,Homo sapiens,"tissue: melanoma, nodal metastasis; cell type: Cells were sort-purified as l...",RNA-Seq,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."
9,GSM8774027,K374_Tumor1_TCR,Homo sapiens,"tissue: melanoma, nodal metastasis; cell type: Cells were sort-purified as l...",OTHER,"Cell Ranger analysis pipelines (version 3.0.2, 10X Genomics) were used to pr..."


### GSE148190: Single cell RNA and TCR sequencing of (human) tumor-infiltrating lymphocytes from human melanoma

https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE148190

Tasks using Python and scVelo, download GSE148190 data

### Plan (reviwed by Claude-Science Opus 5)

1. Step 1 cannot feed step 6. GEO gives you CellRanger matrices — **spliced counts** only. **scVelo needs an unspliced layer**, and GSE148190 deposits no loom/velocyto output and no BAMs. So **"download GSE148190" has to mean download FASTQ and re-quantify with intron-aware counting** (kb count --workflow nac), not download the matrices. That single point reorders the whole pipeline.

2. Scope has to be decided before downloading, not after. **All six GEX libraries are 1.15 Tbases** (~279 GB for the three K409 ones alone). There's no remote compute configured and 444 GB of local disk, so I scoped it to patient K409 — cutaneous primary tumour, involved lymph node, blood — two of four lanes each (140 GB, ~360M read pairs/library, which saturates a 10x channel). One patient across three sites also removes the cross-patient batch effect from the velocity step entirely.

3. "QC we already have" — we don't. Everything so far has been metadata curation. No cell has been counted yet, so mito%, complexity, doublets and the unspliced fraction all still need to be computed, and the unspliced fraction is a QC axis that only exists after re-quantification.

4. "PCA correlation checks": variance-explained elbow plus Spearman correlation of PC1–30 against total counts, genes, mito%, unspliced fraction and library/tissue — the output being a decision on whether the neighbour graph needs **Harmony correction**. That decision has to be made before UMAP/Leiden, because the same graph is reused for scVelo's moments.

5. Top-6 HVGs is a weak fifth step. In FACS-sorted lymphocytes the dispersion-ranked genes are routinely ribosomal, mitochondrial, heat-shock or clonal TCR V-segments, i.e. not cell-state markers. I kept your panel but paired it with canonical T-cell markers so the comparison is visible rather than assumed.
Velocity needs a falsifiable expectation. These are terminally differentiated/exhausted T cells, where splicing-kinetics assumptions are weakest. Stated up front: arrows should run naive/memory → effector → exhausted (the source paper's LAYN⁺ population being the terminal state). If the field is incoherent or confidence is low, that gets reported as the result rather than tuned until it looks plausible.

6. Missing from the original: doublet detection, a check that the sort actually excluded non-T cells, cycling cells as a competing velocity axis, an h5ad checkpoint between stages, and velocity diagnostics (confidence, latent time, PAGA) rather than just a stream plot.

Two things you should know as facts about the data rather than choices: 
  1. velocity here will be on T cells only — every library is CD45⁺CD3⁺ sorted, 
  2. so no malignant melanoma cells exist in this dataset — and 
  3. this series carries no treatment-response labels, so the responder/non-responder axis from the earlier filtering is not available in GSE148190.

Downloads are running now (140 GB, ~3 h at the throughput ENA is giving); the Python 3.12 environment is building in parallel.

Next, implement using Python 3.12

### Harmony correction


[Harmony (R) github](https://github.com/immunogenomics/harmony)

[Detailed Walkthrough of Harmony Algorithm](https://pmc.ncbi.nlm.nih.gov/articles/instance/6884693/bin/NIHMS1539299-supplement-5.html)

[Local Inverse Simpson’s Index measures Mixing and Accuracy](https://pmc.ncbi.nlm.nih.gov/articles/instance/6884693/bin/NIHMS1539299-supplement-6.html)

[Simulated dataset integration](https://pmc.ncbi.nlm.nih.gov/articles/instance/6884693/bin/NIHMS1539299-supplement-7.html)


Harmony (Korsunsky et al., *Nature Methods* 2019) is a batch-integration method that operates on a **low-dimensional embedding** — usually the PCA of log-normalised expression — rather than on the expression matrix itself. You hand it a cell × PC matrix plus one or more batch labels, and it returns a corrected cell × PC matrix that downstream steps (neighbour graph, clustering, UMAP) use in place of the original PCs.

**The algorithm iterates two steps until assignments stabilise:**

1. **Soft clustering with a diversity penalty.** Cells are soft-assigned to *k* clusters by a variant of k-means whose objective includes a term penalising clusters dominated by one batch. The penalty strength is `theta` — higher values push harder toward clusters containing cells from every batch. Soft (fractional) assignment matters: a cell sitting between two states contributes to both, which makes the correction less brittle than hard-cluster approaches.
2. **Cluster-specific linear correction.** Within each cluster, a ridge regression of the embedding on the batch indicators gives per-batch offset vectors, weighted by each cell's soft assignment. Subtracting them moves batches onto each other *locally* — the correction a cell receives depends on which clusters it belongs to, so different cell types can be corrected differently. That locality is the main advantage over a global linear adjustment like ComBat.

Because each round is a few matrix operations in ~20–50 dimensions, it converges in seconds to minutes even at 10⁵–10⁶ cells, which is why it became a default in Scanpy and Seurat pipelines.

**What it does not give you.** Harmony corrects coordinates, not counts. There is no batch-corrected expression matrix to run differential expression on — DE is still done on the uncorrected (normalised) values with the batch as a covariate in the model. If you want a generative model that yields corrected expression and handles counts directly, that is scVI/scANVI territory; fastMNN and Scanorama also sit in the embedding-correction family, and BBKNN corrects the graph instead of the embedding.

**The failure mode that matters, and the one that applied here.** Harmony removes variation associated with the variable you name, *within* local neighbourhoods. That is exactly right when the variable is technical nuisance — different 10x runs, different days, different donors in a study where donor identity is not the question. It is exactly wrong when the variable is confounded with the biology you are trying to measure, because the method has no way to tell the two apart.

That is why I did not apply it in the K409 analysis. The three libraries are three anatomical sites — primary tumour, lymph node, blood — from one patient, prepared with the same 5′ chemistry. "Library" and "tissue of origin" are the same variable, and PC1 (ribosomal-high naive blood cells versus activated tissue T cells, rho = −0.86 with percent-ribosomal) is the naive-to-activated axis the analysis exists to study. Harmonising over library would have mixed blood into tumour and deleted that axis, and the automated rule I had written — correct whenever a batch indicator correlates above 0.3 with a leading PC — would have done it silently. A correlation between a leading PC and a batch label tells you the variable structures the data; it cannot tell you whether that structure is artefact or signal.

Two practical notes. `theta` is worth treating as a real parameter rather than a default when batches have genuinely different cell-type composition — aggressive mixing forces cells of one type onto another when a type is absent from a batch. And in a velocity workflow the choice propagates further than usual: the corrected embedding defines the KNN graph, scVelo's first- and second-order moments are computed over that graph, so batch correction silently changes the smoothing neighbourhood every velocity estimate is built from — one more reason to apply it only when the variable is genuinely a nuisance.

Reference: Korsunsky I, Millard N, Fan J, Slowikowski K, Zhang F, Wei K, Baglaenko Y, Brenner M, Loh PR, Raychaudhuri S. Fast, sensitive and accurate integration of single-cell data with Harmony. Nat Methods. 2019 Dec;16(12):1289-1296. doi: 10.1038/s41592-019-0619-0. Epub 2019 Nov 18. PMID: 31740819; PMCID: PMC6884693.

### Reading strand option docs and preparing subsamples

1. **kallisto** performs pseudoalignment, a method that assigns reads to transcripts without fully aligning them. This enables extremely fast and accurate transcript quantification.

2. **bustools** processes barcode and UMI information and provides utilities for manipulating BUS—a compact representation of barcodes, UMIs, and transcript equivalence classes.

3. **kb-python** offers high-level workflows that automate common tasks using kallisto and bustools, including reference generation and the processing of both bulk and single-cell RNA-seq experiments. It automatically handles downloading and formatting reference transcriptomes, manages file organization, and ensures that kallisto and bustools are invoked with consistent parameters. **kb-python** is a python package for processing single-cell RNA-sequencing. It wraps the kallisto | bustools single-cell RNA-seq command line tools in order to unify multiple processing workflows.

kb-python was first developed by Kyung Hoi (Joseph) Min and A. Sina Booeshaghi while in Lior Pachter's lab at Caltech.  
Ref: Melsted, P., Booeshaghi, A.S., et al. 
Modular, efficient and constant-memory single-cell RNA-seq preprocessing. 
Nat Biotechnol  39, 813–818 (2021). 

https://doi.org/10.1038/s41587-021-00870-2

github: https://github.com/pachterlab/kb_python

```Bash
uv pip install -U kb-python
```

### Bustools

https://kallisto.readthedocs.io/en/latest/

bustools is a program for manipulating BUS files for single cell RNA-Seq datasets. It can be used to error correct barcodes, collapse UMIs, produce gene count or transcript compatibility count matrices, and is useful for many other tasks. See the kallisto | bustools website for examples and instructions on how to use bustools as part of a single-cell RNA-seq workflow.

https://github.com/BUStools/bustools/

Ref: Melsted, P., Booeshaghi, A.S., Liu, L. et al. Modular, efficient and constant-memory single-cell RNA-seq preprocessing. Nat Biotechnol 39, 813–818 (2021). https://doi.org/10.1038/s41587-021-00870-2


kb is installed and new enough: `--strand`, `--workflow nac` and `--h5ad` are all there. For RNA velocity in current kb-python, **`nac`** (nascent/ambiguous/mature) is the workflow to use. It replaces the old `lamanno` workflow.

The `kbref` line printed nothing, so `$WS/kbref/` is either missing or empty. Check which:


```bash
WS=/home/flavio/.claude-science/orgs/abb023bb-bd59-418e-bf97-0ebfba525efa/workspaces/34193696-f813-4ebc-adf4-6f55b355bc9a
WSuv=/home/flavio/uv/rna_velocity/download/GSE148190

ls -la "$WS"/kbref 2>&1 | head
```

If it's missing, build the nac reference. The quickest route is kb's prebuilt download (human or mouse):


### kb ref tables:

1. index: index.idx 
2. transcript to gene (ENST to ENTSG): t2g.txt 
3. cdna: cdna_t2c.txt 
4. nascent RNA: nascent_t2c.txt


#### CLI:

> -g T2G:    Path to transcript-to-gene mapping to be generated  
> \-i INDEX: Path to the kallisto index to be constructed.  
> -c1 T2C:   Path to generate cDNA transcripts-to-capture  
> -c2 T2C:   Path to generate unprocessed transcripts-to-capture  

```bash
mkdir -p "$WS"/kbref && cd "$WS"/kbref

WSuv=/home/flavio/uv/rna_velocity/download/GSE148190
cd $WSuv

kb ref -d human --workflow nac \
  -i index.idx -g t2g.txt -c1 cdna_t2c.txt -c2 nascent_t2c.txt # --overwrite

ls -lstrh   # expect index.idx (several GB), t2g.txt, cdna_t2c.txt, nascent_t2c.txt

```

or a custom Ensembl genome + GTF (slower, needs about 32 GB RAM for human):

```bash

find ~/uv/ -iname "Homo_sapiens.GRCh38*.gtf*" 2>/dev/null

wget -q --show-progress https://ftp.ensembl.org/pub/release-112/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz
wget -q --show-progress https://ftp.ensembl.org/pub/release-112/gtf/homo_sapiens/Homo_sapiens.GRCh38.112.gtf.gz


kb ref --workflow nac -t 8 \
  -i index.idx -g t2g.txt -c1 cdna_t2c.txt -c2 nascent_t2c.txt \
  -f1 cdna.fa -f2 nascent.fa \
  Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz Homo_sapiens.GRCh38.112.gtf.gz


SRR=SRR11492025
N=1000000
mkdir -p "$WS"/subsample
for r in 1 2; do
  zcat "$WSuv"/fastq/${SRR}_${r}.fastq.gz 2>/dev/null | head -n $((N*4)) | gzip -1 > "$WSuv"/subsample/sub_${r}.fastq.gz
done

# pairing + length check
for r in 1 2; do
  echo "R$r reads: $(( $(zcat "$WSuv"/subsample/sub_${r}.fastq.gz | wc -l) / 4 ))"
  zcat "$WSuv"/subsample/sub_${r}.fastq.gz | awk 'NR%4==2{print length($0)} NR>=40000{exit}' | sort -n | uniq -c | sort -k1,1nr | head -3
done

# result
R1 reads: 1000000
  10000 150
R2 reads: 1000000
  10000 150

```

### Both mates are 150 bp, so the "28 bp → v3" shortcut doesn't apply. 

**This is common:**

many labs sequence 10x libraries PE150 on NovaSeq. 
- R1 still carries the barcode+UMI in its first 26–28 bp, followed by the poly-T and some cDNA. 
- kallisto only reads the positions defined by `-x` 
  - for 10xv3: 
    - barcode at R1 0–16, 
    - UMI at 16–28, 
    - cDNA from R2,
    - so 150 bp reads are fine. But you now need a different way to tell v2 from v3, and to confirm the file is 10x at all.

**Check where the poly-T starts in R1.** It should begin right after the barcode+UMI: at position 27 for v2 (16+10) or 29 for v3 (16+12).

```bash
for r in 1 2; do
  echo "== R$r: poly-T (TTTTTTTTTT) start position =="
  zcat "$WSuv"/subsample/sub_${r}.fastq.gz | awk 'NR%4==2' | head -100000 \
    | awk '{p=index($0,"TTTTTTTTTT"); if(p>0) print p}' | sort -n | uniq -c | sort -k1,1nr | head -5
done
```

#### result
```bash
== R1: poly-T (TTTTTTTTTT) start position ==
     13 125
     11 124
     11 126
      9 114
      8 108
== R2: poly-T (TTTTTTTTTT) start position ==
   1354 26
    131 24
    117 25
     41 23
     29 22

```

**R2 is the barcode+UMI read.** It has the poly-T signal and R1 has essentially none. The SRA `_1`/`_2` files are swapped relative to 10x convention, so kb must get them in the order **`sub_2` then `sub_1`**. Keeping the original order would take the barcodes from a cDNA read, and almost nothing would land on the whitelist.

**The chemistry is still ambiguous.** Poly-T normally begins at position 27 (v2) or 29 (v3), and a peak at 26 or earlier fits neither exactly. The hit rate is also low: only about 1.4% of reads have 10 perfect Ts. That can happen with 150 bp barcode reads, because quality drops inside homopolymers. The whitelist test settles it, so run both chemistries with the corrected mate order:
|
```bash
for chem in 10xv2 10xv3; do
  kb count --workflow nac -x $chem -t 8 --h5ad -o kb_sub_$chem \
    -i $root_kbref/index.idx -g $root_kbref/t2g.txt \
    -c1 $root_kbref/cdna_t2c.txt -c2 $root_kbref/nascent_t2c.txt \
    "$WS"/subsample/sub_2.fastq.gz "$WS"/subsample/sub_1.fastq.gz
done

python - <<'EOF'
import json
for d in ["kb_sub_10xv2", "kb_sub_10xv3"]:
    ri = json.load(open(f"{d}/run_info.json"))
    ins = json.load(open(f"{d}/inspect.json"))
    print(f"{d}: pseudoaligned {ri['p_pseudoaligned']}% | on whitelist {ins.get('percentageReadsOnWhitelist')}%")
EOF
```

* **One chemistry above about 80% on the whitelist:** that's your chemistry.
* **Both low:** the data probably isn't 10x 3' Chromium (it could be 5', Drop-seq or BD Rhapsody). Check the library metadata:

```bash
curl -s "https://www.ebi.ac.uk/ena/portal/api/filereport?accession=SRR11492025&result=read_run&fields=study_accession,experiment_title,library_strategy,library_layout,instrument_model,read_count&format=tsv"

# run_accession	study_accession	experiment_title	library_strategy	library_layout	instrument_model	read_count
# SRR11492025	PRJNA623409	Illumina NovaSeq 6000 sequencing: GSM4455935: melanoma_K409_tumor_GEX Homo sapiens RNA-Seq	RNA-Seq	PAIRED	Illumina NovaSeq 6000	181375026
```

The `experiment_title`, or the linked GEO series, normally names the kit, for example "Chromium Single Cell 3' v3".

### Metadata: the sample is **GSM4455935, "melanoma_K409_tumor_GEX"**, human, NovaSeq PE150

1. about 181M read pairs.
2. The **"GEX"** suffix is 10x's name for the gene-expression library in **5' Chromium + V(D)J** experiments, 
  - where each sample also has TCR/BCR libraries.
3. If this is 5' data, it explains the odd poly-T result:
  - 5' libraries have no poly-T right after the barcode+UMI. 
  - Instead, the **TSO** (`TTTCTTATATGGG`) sits at position 27, directly after the 16 bp barcode and 10 bp UMI.

Check for the TSO in both mates:

```bash
for r in 1 2; do
  echo "== R$r: TSO start position =="
  zcat "$WS"/subsample/sub_${r}.fastq.gz | awk 'NR%4==2' | head -100000 \
    | awk '{p=index($0,"TTTCTTATATGGG"); if(p>0) print p}' | sort -n | uniq -c | sort -k1,1nr | head -3
done

== R1: TSO start position ==
  69982 27
   1655 26
   1109 28
== R2: TSO start position ==
     10 49
      5 32
      4 10
```

That settles it. R1 has the TSO at position 27 in about 70% of reads, so this is **10x 5' v2**, and **R1 is the barcode+UMI read**.

That reverses what I said earlier: the mates are not swapped. The poly-T at position 26 in R2 wasn't a barcode-read signature. It was probably R2 reading into the poly-A tail at the end of short fragments. So the order is sub_1 (barcode) then sub_2 (cDNA).

Run the strand test on the subsample:

```Bash

BC="$WS"/subsample/sub_1.fastq.gz
CD="$WS"/subsample/sub_2.fastq.gz
for s in forward reverse unstranded; do
  kb count --workflow nac -x 10xv2 --strand $s -t 8 --h5ad -o kb_sub_5p_$s \
    -i $root_kbref/index.idx -g $root_kbref/t2g.txt \
    -c1 $root_kbref/cdna_t2c.txt -c2 $root_kbref/nascent_t2c.txt \
    "$BC" "$CD"
done

python - <<'EOF'
import json
for s in ["forward","reverse","unstranded"]:
    d=f"kb_sub_5p_{s}"
    ri=json.load(open(f"{d}/run_info.json")); ins=json.load(open(f"{d}/inspect.json"))
    print(f"{s:10s} pseudoaligned {ri['p_pseudoaligned']}% | on whitelist {ins.get('percentageReadsOnWhitelist')}%")
EOF
```

### Checking forward, reverse, unstranded (in "$WS"/subsample)

In [ ]:
import json

WS = '~/.claude-science/orgs/abb023bb-bd59-418e-bf97-0ebfba525efa/workspaces/34193696-f813-4ebc-adf4-6f55b355bc9a'

for s in ["forward","reverse","unstranded"]:
    d=f"kb_sub_5p_{s}"
    ri=json.load(open(f"{d}/run_info.json")); ins=json.load(open(f"{d}/inspect.json"))
    print(f"{s:10s} pseudoaligned {ri['p_pseudoaligned']}% | on whitelist {ins.get('percentageReadsOnWhitelist')}%")
EOF


How to read the result:

| Result | Meaning |
|---|---|
| R1 peak at **29**, R2 has none | 10x v3 → `-x 10xv3` |
| R1 peak at **27**, R2 has none | 10x v2 → `-x 10xv2` |
| Peak in **R2**, not R1 | mates swapped, so pass R2 first to kb |
| No clear peak anywhere | not 10x 3' (could be 5', Smart-seq or bulk), so check the SRA metadata |

A clear peak in R1 usually covers a large fraction of the 100k reads.

**Confirm empirically on the subsample.** Run v3 first, then v2 if it looks poor, and compare:

```bash
kb count --workflow nac -x 10xv3 -t 8 --h5ad -o kb_sub_v3 \
  -i $root_kbref/index.idx -g $root_kbref/t2g.txt -c1 $root_kbref/cdna_t2c.txt -c2 $root_kbref/nascent_t2c.txt \
  "$WS"/subsample/sub_1.fastq.gz "$WS"/subsample/sub_2.fastq.gz

python - <<'EOF'
import json
for d in ["kb_sub_v3"]:
    ri = json.load(open(f"{d}/run_info.json"))
    ins = json.load(open(f"{d}/inspect.json"))
    print(d, "pseudoaligned %:", ri["p_pseudoaligned"],
          "| reads on whitelist %:", ins.get("percentageReadsOnWhitelist"))
EOF
```

The correct chemistry gives a high on-whitelist rate, typically over 90%. The wrong one drops to around 0–10%, because v2 and v3 use different barcode whitelists.

### A few notes on that command:

1. -x: set it from the R1 length check
  - 10xv2 for 26 bp or
  - 10xv3 for 28 bp.

2. --strand: leave it unset. 

For 10x the default is already forward, which you can confirm with kb --list.
Output: the h5ad in kb_sub/counts_unfiltered/ has mature, nascent and ambiguous layers. 

For **scVelo**, use **mature as spliced** and **nascent** as **unspliced**.  
  - Mapping rate: check kb_sub/run_info.json for p_pseudoaligned. 
  - For 10x against the nac index you'd expect roughly 50–80%. 
  - A very low value usually means the wrong species or the wrong -x.

organism is SRR11492025? >> `-d human`

### Test run on the 1M-pair subsample:

```bash

cd ~/uv/rna_velocity

WS=/home/flavio/.claude-science/orgs/abb023bb-bd59-418e-bf97-0ebfba525efa/workspaces/34193696-f813-4ebc-adf4-6f55b355bc9a
root_kbref="$WS"/kbref
echo $root_kbref

ls -ls $root_kbref
# ls -la $root_kbref 2>&1 | head

cd ~/uv/rna_velocity
source .venv/bin/activate

kb count --workflow nac -x 10xv3 -t 8 --h5ad -o kb_sub \
  -i $root_kbref/index.idx -g $root_kbref/t2g.txt -c1 $root_kbref/cdna_t2c.txt -c2 $root_kbref/nascent_t2c.txt \
  "$WS"/subsample/sub_1.fastq.gz  "$WS"/subsample/sub_2.fastq.gz

python -c "import json; d=json.load(open('kb_sub/run_info.json')); print(d['n_processed'], d['p_pseudoaligned'])"
```

### Reading strand option docs and preparing subsamples - after installation

```Bash
FQ="$WS/fastq"
SRR=SRR11492025
N=1000000                      # read pairs

kb count --help 2>&1 | grep -A6 -- "--strand"

echo "=== progress ==="
ls -l "$FQ"/*.fastq.gz | awk '{s+=$5} END {printf "fastq %.1f GB (%d files)\n", s/1e9, NR}'
ls -l "$WS"/kbref/ 2>/dev/null | awk 'NR>1 {print $9, $5}'

echo "=== 1M-pair subsample from (possibly partial) files ==="
[[ -s "$FQ/${SRR}_1.fastq.gz" && -s "$FQ/${SRR}_2.fastq.gz" ]] || { echo "missing input in $FQ"; exit 1; }
mkdir -p subsample
for r in 1 2; do
  zcat "$FQ/${SRR}_${r}.fastq.gz" 2>/dev/null | head -n $((N*4)) | gzip -1 > subsample/sub_${r}.fastq.gz
done
ls -l subsample/ | awk 'NR>1 {print $9, $5}'

cho "=== pairing check ==="
n1=$(zcat subsample/sub_1.fastq.gz | wc -l); n2=$(zcat subsample/sub_2.fastq.gz | wc -l)
echo "R1 reads: $((n1/4))   R2 reads: $((n2/4))"
[[ $n1 -eq $n2 ]] || echo "WARNING: unequal read counts, pairs are out of sync"
paste <(zcat subsample/sub_1.fastq.gz | awk 'NR%4==1{print $1}' | head -3) \
      <(zcat subsample/sub_2.fastq.gz | awk 'NR%4==1{print $1}' | head -3)

echo "=== read lengths (first 10k reads) ==="
for r in 1 2; do
  echo "R$r:"; zcat subsample/sub_${r}.fastq.gz | awk 'NR%4==2{print length($0)} NR>=40000{exit}' | sort -n | uniq -c | sort -k1,1nr | head -5
done
```